# GOES Energetic Particle Detectors — Implementation / 구현

**Paper**: Onsager, T. G., R. Grubb, J. Kunches, L. Matheson, D. Speich, R. Zwickl, and H. Sauer, "Operational uses of the GOES energetic particle detectors," *Proc. SPIE* **2812**, 281–290 (1996). DOI: 10.1117/12.254075

This notebook implements three operational components from the paper:
1. The **NOAA SEC alert/warning logic** — boolean alert evaluation against the 10/1/1000 pfu thresholds.
2. The **NOAA Solar Radiation Storm (S-scale) classifier** — the post-1996 logarithmic extension that builds directly on Table 2 thresholds.
3. The **operational data-processing pipeline** — GCR background subtraction (10-day rolling minimum with safeguard) and a synthetic SEP event detector.

이 노트북은 논문의 세 가지 운영 구성 요소를 구현합니다: NOAA SEC alert/warning 논리, NOAA S-스케일 분류기, 그리고 GCR 배경 제거 파이프라인을 포함한 합성 SEP 사건 검출기.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from dataclasses import dataclass
from typing import Optional

plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['font.size'] = 11
rng = np.random.default_rng(seed=20260425)

## Part 1: NOAA SEC Alert/Warning Logic / NOAA SEC 알림 이해

Implements the operational thresholds documented in Table 2 of Onsager et al. (1996):

- **Proton alert**: `J(>10 MeV) > 10 pfu` **OR** `J(>100 MeV) > 1 pfu`
- **Electron alert**: `J(>2 MeV) > 1000 pfu`
- 1 pfu = 1 particle / (cm² s sr)

Onsager et al.(1996)의 Table 2에 명시된 운영 임계값을 구현합니다.

In [ ]:
@dataclass
class NOAAThresholds:
    """NOAA SEC alert/warning thresholds from Onsager et al. (1996), Table 2.

    Attributes:
        proton_10mev_pfu: Threshold for the >10 MeV integral proton alert (pfu).
        proton_100mev_pfu: Threshold for the >100 MeV integral proton alert (pfu).
        electron_2mev_pfu: Threshold for the >2 MeV integral electron alert (pfu).
    """
    proton_10mev_pfu: float = 10.0
    proton_100mev_pfu: float = 1.0
    electron_2mev_pfu: float = 1000.0


def evaluate_alerts(j_p10: float,
                    j_p100: float,
                    j_e2: float,
                    th: NOAAThresholds = NOAAThresholds()) -> dict:
    """Evaluate NOAA SEC observed-conditions alerts.

    Args:
        j_p10: Integral proton flux above 10 MeV in pfu.
        j_p100: Integral proton flux above 100 MeV in pfu.
        j_e2: Integral electron flux above 2 MeV in pfu.
        th: Threshold container.

    Returns:
        A dictionary of boolean alerts keyed by their canonical names.
    """
    proton_alert = (j_p10 > th.proton_10mev_pfu) or (j_p100 > th.proton_100mev_pfu)
    electron_alert = j_e2 > th.electron_2mev_pfu
    return {
        'proton_alert': proton_alert,
        'electron_alert': electron_alert,
        'proton_10MeV_exceeds': j_p10 > th.proton_10mev_pfu,
        'proton_100MeV_exceeds': j_p100 > th.proton_100mev_pfu,
    }


# Worked example: hard-spectrum SEP triggers >100 MeV alert first.
example = evaluate_alerts(j_p10=8.5, j_p100=1.4, j_e2=200.0)
print('Example (hard-spectrum SEP, J>10 MeV=8.5, J>100 MeV=1.4):')
for k, v in example.items():
    print(f'  {k}: {v}')

## Part 2: NOAA Solar Radiation Storm (S-scale) Classifier / NOAA 태양 방사선 폭풍 S-스케일 분류기

The NOAA Space Weather Scales (formalized in 2003) are the logarithmic operational extension of the Onsager et al. (1996) `>10 MeV @ 10 pfu` threshold. The S-scale assigns:

| S level | J(>10 MeV) (pfu) | Description |
|---|---|---|
| S0 | < 10 | Background |
| S1 | ≥ 10 | Minor |
| S2 | ≥ 100 | Moderate |
| S3 | ≥ 1000 | Strong |
| S4 | ≥ 10⁴ | Severe |
| S5 | ≥ 10⁵ | Extreme |

NOAA Space Weather Scales는 Onsager et al.(1996) 임계값 10 pfu @ >10 MeV을 로그 빈으로 확장한 운영 체계입니다.

In [ ]:
S_SCALE_TABLE = [
    (0, 1e1, 'S0', 'Background'),
    (1, 1e2, 'S1', 'Minor'),
    (2, 1e3, 'S2', 'Moderate'),
    (3, 1e4, 'S3', 'Strong'),
    (4, 1e5, 'S4', 'Severe'),
    (5, np.inf, 'S5', 'Extreme'),
]


def classify_s_scale(j_p10: np.ndarray) -> np.ndarray:
    """Classify >10 MeV proton flux into NOAA S-scale levels (S0-S5).

    Args:
        j_p10: Array of integral proton flux above 10 MeV in pfu.

    Returns:
        Integer array of S-levels (0 through 5), same shape as input.
    """
    j_p10 = np.atleast_1d(np.asarray(j_p10, dtype=float))
    levels = np.zeros_like(j_p10, dtype=int)
    levels[j_p10 >= 1e1] = 1
    levels[j_p10 >= 1e2] = 2
    levels[j_p10 >= 1e3] = 3
    levels[j_p10 >= 1e4] = 4
    levels[j_p10 >= 1e5] = 5
    return levels


def s_scale_label(level: int) -> str:
    """Return human-readable label for an S-scale integer level."""
    for lvl, _, name, desc in S_SCALE_TABLE:
        if lvl == level:
            return f'{name} ({desc})'
    return 'unknown'


# Demonstrate on a sweep of fluxes.
test_fluxes = np.array([0.5, 5.0, 12.0, 200.0, 5000.0, 5e4, 5e5])
test_levels = classify_s_scale(test_fluxes)
print('Flux (pfu) -> S-scale')
for f, lv in zip(test_fluxes, test_levels):
    print(f'  {f:>10.1f}  ->  {s_scale_label(lv)}')

## Part 3: GCR Background Subtraction & SEP Detection / GCR 배경 제거와 SEP 검출

Implements the **10-day low-pass minimum filter with latch-prevention safeguard** described in §2 of the paper. Then synthesizes a multi-day proton time series with quiet GCR + a 20-October-1995-style SEP event, and runs the alert pipeline on it.

논문 §2에서 설명한 10일 상시 최소값 저역 통과 필터(락킩 방지 안전장치 포함)를 구현하고, 합성 SEP 시계열에 적용하여 alert 파이프라인을 검증합니다.

In [ ]:
def rolling_min_background(rate: np.ndarray,
                           window_samples: int,
                           tolerance_factor: float = 3.0) -> np.ndarray:
    """Compute the GCR-background estimate per Onsager et al. (1996), §2.

    Implements a rolling-minimum low-pass with a latch-prevention safeguard:
    a new minimum is adopted only if it lies within ``tolerance_factor`` of
    the previous minimum; otherwise the previous estimate is reused. This
    prevents the algorithm from anchoring to an SEP-elevated minimum during
    multi-day events.

    Args:
        rate: Time series of count rate (or flux) samples.
        window_samples: Number of samples in the trailing window
            (e.g., 10 days at 5-minute cadence -> 2880 samples).
        tolerance_factor: Multiplicative tolerance for adopting a new minimum.
            A new candidate ``B_c`` replaces the previous ``B_prev`` only when
            ``B_prev / tolerance_factor <= B_c <= B_prev * tolerance_factor``.

    Returns:
        Background-estimate array of the same length as ``rate``.
    """
    n = len(rate)
    bkg = np.zeros(n, dtype=float)
    # Initialize with the minimum of the first window.
    init_window = rate[: max(1, window_samples)]
    bkg[0] = np.min(init_window)
    for i in range(1, n):
        lo = max(0, i - window_samples + 1)
        candidate = float(np.min(rate[lo : i + 1]))
        prev = bkg[i - 1]
        within = (prev / tolerance_factor <= candidate <= prev * tolerance_factor)
        bkg[i] = candidate if within else prev
    return bkg


# Build a synthetic 30-day proton time series at 5-minute cadence.
samples_per_day = 24 * 12  # 5-minute cadence
n_days = 30
n = samples_per_day * n_days
t_days = np.arange(n) / samples_per_day  # days since start

# Quiet GCR background ≈ 0.05 pfu in J(>10 MeV) with mild diurnal/random fluctuation.
gcr = 0.05 + 0.005 * np.sin(2 * np.pi * t_days) + 0.003 * rng.standard_normal(n)
gcr = np.clip(gcr, 1e-3, None)

# SEP event: rapid rise on day 20, exponential decay over ~5 days, peaks at 80 pfu.
sep = np.zeros(n)
onset_day = 20.0
rise_tau = 0.25  # days
decay_tau = 2.0  # days
peak_pfu = 80.0
mask = t_days >= onset_day
dt = t_days[mask] - onset_day
sep[mask] = peak_pfu * (1 - np.exp(-dt / rise_tau)) * np.exp(-dt / decay_tau)

# Total observed flux = background + SEP
flux_p10 = gcr + sep

# Run the background filter (10-day window).
window = 10 * samples_per_day
bkg_est = rolling_min_background(flux_p10, window_samples=window, tolerance_factor=3.0)
flux_corrected = np.maximum(flux_p10 - bkg_est, 0.0)

print(f'Synthesized {n_days} days at 5-min cadence ({n} samples).')
print(f'GCR mean: {gcr.mean():.4f} pfu | SEP peak: {sep.max():.2f} pfu')
print(f'Background-estimate mean before onset: {bkg_est[: int(onset_day*samples_per_day)].mean():.4f} pfu')
print(f'Background-estimate mean after onset:  {bkg_est[int(onset_day*samples_per_day):].mean():.4f} pfu')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax = axes[0]
ax.semilogy(t_days, flux_p10, color='k', lw=0.6, label='Observed J(>10 MeV)')
ax.semilogy(t_days, bkg_est, color='C1', lw=1.4, label='10-day low-pass background')
ax.axhline(10.0, color='C3', ls='--', lw=1.0, label='S1 / NOAA alert threshold (10 pfu)')
ax.axhline(100.0, color='C2', ls='--', lw=1.0, label='S2 (100 pfu)')
ax.set_ylabel('Flux / pfu')
ax.set_title('Synthetic GOES >10 MeV proton flux with SEP event onset day 20')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, which='both', alpha=0.3)

# Color-code the S-scale level along the time axis.
levels = classify_s_scale(flux_corrected)
ax2 = axes[1]
colors = ['#dddddd', '#fff2cc', '#ffd966', '#f6b26b', '#cc4125', '#660000']
for lvl in range(6):
    sel = (levels == lvl)
    if np.any(sel):
        ax2.fill_between(t_days, 0, 1, where=sel, color=colors[lvl],
                         step='mid', label=f'S{lvl}')
ax2.set_xlabel('Days since start of month')
ax2.set_ylabel('S-scale')
ax2.set_yticks([])
ax2.set_ylim(0, 1)
ax2.legend(loc='upper left', ncol=6, fontsize=9)
ax2.set_title('NOAA S-scale classification of background-corrected flux')

plt.tight_layout()
plt.show()

In [ ]:
def detect_sep_events(t_days: np.ndarray,
                      j_p10_corr: np.ndarray,
                      threshold_pfu: float = 10.0,
                      min_duration_samples: int = 3) -> pd.DataFrame:
    """Detect contiguous intervals where corrected flux exceeds the alert threshold.

    Args:
        t_days: Time stamp array (days since reference epoch).
        j_p10_corr: Background-corrected J(>10 MeV) flux in pfu.
        threshold_pfu: Alert threshold (defaults to 10 pfu = NOAA S1 / 1996 alert).
        min_duration_samples: Minimum consecutive samples required for an event
            to qualify (suppresses single-sample noise excursions).

    Returns:
        DataFrame with one row per detected event and columns
        ``onset_day``, ``end_day``, ``duration_days``, ``peak_pfu``, ``s_scale``.
    """
    above = j_p10_corr > threshold_pfu
    events = []
    i = 0
    n = len(above)
    while i < n:
        if above[i]:
            j = i
            while j < n and above[j]:
                j += 1
            if (j - i) >= min_duration_samples:
                seg = j_p10_corr[i:j]
                events.append({
                    'onset_day': float(t_days[i]),
                    'end_day': float(t_days[j - 1]),
                    'duration_days': float(t_days[j - 1] - t_days[i]),
                    'peak_pfu': float(seg.max()),
                    's_scale': int(classify_s_scale(np.array([seg.max()]))[0]),
                })
            i = j
        else:
            i += 1
    return pd.DataFrame(events)


events_df = detect_sep_events(t_days, flux_corrected, threshold_pfu=10.0)
print('Detected SEP events (>10 pfu, S1+):')
print(events_df.to_string(index=False))

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Proton alert threshold / 양성자 알림 임계값 | >10 MeV @ 10 pfu, >100 MeV @ 1 pfu | NOAA S1 (S-scale anchor) / NOAA S1 기준 |
| Electron alert / 전자 알림 | >2 MeV @ 1000 pfu | Still in operational use at SWPC / SWPC 현재 운영 중 |
| GCR background subtraction / GCR 배경 제거 | 10-day rolling minimum + latch-prevention | Same algorithm in GOES-R SGPS pipeline / GOES-R SGPS도 동일 |
| Channel response factor / 채널 응답 인자 | $R = G\Delta E$ from Panametrics calibration | Inter-calibrated by Rodriguez et al. 2014 / Rodriguez et al. 2014에서 교차보정 |
| S-scale extension / S-스케일 확장 | Implicit (Table 2) | Formalized in NOAA Space Weather Scales 2003 / 2003 NOAA 스케일 공식화 |

The synthetic example reproduces the operational logic that NOAA SWPC has used continuously since 1996: a 10-day rolling-minimum subtraction isolates SEP events from the slowly varying GCR background; the alert threshold (10 pfu) flags the event; and the S-scale classifier maps the peak flux to a public-facing severity tag.

합성 예제는 1996년부터 NOAA SWPC가 지속적으로 사용해 온 운영 논리를 그대로 재현합니다: 10일 상시 최소값 제거로 GCR 배경을 분리하고, 알림 임계값(10 pfu)으로 사건을 표시하며, S-스케일 분류기가 최대 플럭스를 공식적 심각도 태그로 매핑합니다.